# NVIDIA Nemotron Reasoning Challenge — Starter Notebook

**A minimal working example** that loads the model, trains a LoRA adapter on the competition data, and packages a valid submission.

### Required Kaggle Inputs (4)
1. **Competition data** — `nvidia-nemotron-model-reasoning-challenge`
2. **Model** — `metric/nemotron-3-nano-30b-a3b-bf16` (Transformers format)
3. **Offline packages** — `dennisfong/nvidia-nemotron-offline-packages`
4. **NVIDIA utility script** — `ryanholbrook/nvidia_utility_script` (attached as Kaggle utility script)

### What this notebook does
1. Fixes the Kaggle environment (Triton permissions, sklearn/scipy conflict)
2. Installs `datasets` and `trl` from offline wheels
3. Loads the Nemotron 3 Nano 30B model
4. Discovers the 12 sensitive layers (per the [Nemotron 3 Nano report §4.2](https://arxiv.org/abs/2512.20848))
5. Applies LoRA (rank 32) to those layers only — produces a ~26 MB adapter, not 3+ GB
6. Trains on all 9,500 examples with SFTTrainer
7. Validates on 5 examples
8. Packages `submission.zip`

> **GPU required.** Set Accelerator to **GPU RTX Pro 6000** in notebook settings.

## Step 1: Environment Fixes

> **Updated March 2026:** sklearn blocker removed (fixed in new Docker image). Mamba3/cutlass stub added.

In [1]:
# The Kaggle container has three issues we need to fix before importing anything:
#
# 1. mamba_ssm (required by Nemotron) lives in the utility script, not in site-packages
# 2. ptxas-blackwell binary is read-only — Triton can't execute it
# 3. ~~sklearn/scipy conflict~~ FIXED in new Docker image (March 2026)

import sys, os, shutil, stat

# Fix 1: Add utility script to Python path (provides mamba_ssm, triton, etc.)
sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

# Fix 2: Copy ptxas-blackwell to /tmp with execute permissions
ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    # Copy the whole bin directory
    src_bin = os.path.dirname(ptxas_src)
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

    # Point Triton to the writable copy
    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

# Also patch get_ptxas_version to avoid calling the binary at compile time
import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: '12.0'

# Fix 3: sklearn blocker no longer needed (numpy/scipy fixed in new Docker image)

print('Environment fixes applied.')


Environment fixes applied.


## Step 2: Install Packages from Offline Wheels

In [2]:
# The Kaggle competition notebooks have no internet access.
# datasets and trl are not pre-installed, so we install from offline wheels.
# IMPORTANT: use --no-deps to avoid downgrading numpy (which breaks scipy).

import subprocess

OFFLINE_DIR = '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages'

subprocess.run(
    f'{sys.executable} -m pip install -q --no-index --find-links {OFFLINE_DIR} '
    f'datasets trl --no-deps',
    shell=True
)
# Install lightweight dependencies that may be missing
subprocess.run(
    f'{sys.executable} -m pip install -q --no-index --find-links {OFFLINE_DIR} '
    f'multiprocess dill xxhash 2>/dev/null || true',
    shell=True
)

# Verify everything imports
import datasets
import trl

# Stub out mamba3 (requires cutlass which isn't installed)
# Nemotron only uses Mamba/Mamba2, not Mamba3 — safe to skip
import types
for _mod_name in [
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
]:
    sys.modules[_mod_name] = types.ModuleType(_mod_name)
# Give mamba3 a dummy Mamba3 class so the __init__ import doesn't fail
sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

import mamba_ssm

print(f'datasets: {datasets.__version__}')
print(f'trl:      {trl.__version__}')
print(f'mamba_ssm: {mamba_ssm.__version__}')


datasets: 4.8.3
trl:      0.24.0
mamba_ssm: 2.3.1


## Step 3: Imports and Configuration

In [3]:
import os, re, random, zipfile, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from collections import defaultdict
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# Paths — these must match your Kaggle notebook inputs
MODEL_PATH = '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'
DATA_PATH = '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv'
OUTPUT_DIR = '/kaggle/working/adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
SEED = 42
LORA_RANK = 32        # competition max is 32
MAX_SEQ_LEN = 1024    # fits most prompts + answers
NUM_EPOCHS = 1        # increase if you have time budget
GRAD_ACCUM = 8        # effective batch size = 8
LR = 5e-5             # conservative — small adapter needs gentle learning rate

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


PyTorch: 2.12.0.dev20260324+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Step 4: Load Model and Tokenizer

In [4]:
# The Nemotron model uses a custom Triton rmsnorm kernel that may fail
# on some GPU/driver combos. We provide a pure PyTorch fallback.
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

# Apply the fallback before loading the model
for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer loaded. Vocab size: {len(tokenizer)}')

# Load model
print('Loading Nemotron 3 Nano 30B (this takes ~8 minutes)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print('Model loaded.')

# Force the slow (pure PyTorch) code path for Nemotron-H layers
# The fast CUDA kernels may not work on all container versions
for name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in name and hasattr(mod, 'is_fast_path_available'):
        mod.is_fast_path_available = False
        print(f'Patched {name}: using slow path')

# Re-apply rmsnorm fallback (model loading may import new modules)
for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn') and mod.rmsnorm_fn is not _pure_rmsnorm_fn:
        mod.rmsnorm_fn = _pure_rmsnorm_fn


Tokenizer loaded. Vocab size: 131072
Loading Nemotron 3 Nano 30B (this takes ~8 minutes)...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded.
Patched transformers_modules._1.modeling_nemotron_h: using slow path


## Step 5: Discover Sensitive Layers

Per the [Nemotron 3 Nano report (§4.2)](https://arxiv.org/abs/2512.20848), only **12 of 52 layers** are sensitive to weight changes:
- **6 GQA attention layers** (marked `*` in the architecture pattern)
- **6 Mamba layers** immediately before each attention layer

We target all Linear modules in these 12 layers, skipping Conv1d (incompatible with LoRA), router/gate weights, and routable experts (sparse traffic). This produces a **~20-50 MB adapter** instead of 3+ GB.

In [5]:
# ═══════════════════════════════════════════════════════════
# LAYER TARGETING — based on Nemotron 3 Nano report §4.2
#
# The hybrid_override_pattern tells us which layers are attention (*).
# Only 12 of 52 layers are sensitive to weight perturbations:
#   - 6 GQA attention layers (marked * in pattern)
#   - 6 Mamba layers immediately before each attention layer
#
# We target ALL Linear modules in these 12 layers, except:
#   - Conv1d (groups=6144, crashes LoRA)
#   - Router/gate weights (sparse routing, bad LoRA targets)
#   - Routable experts index >= 2 (sparse traffic, low ROI)
# ═══════════════════════════════════════════════════════════
import re as _re

config = model.config
pattern = getattr(config, 'hybrid_override_pattern', '')
print(f'Architecture pattern: {pattern}')
print(f'Pattern length: {len(pattern)} layers')

# Find sensitive layer indices from the pattern
attn_indices = [i for i, c in enumerate(pattern) if c == '*']
pre_attn_indices = [i - 1 for i in attn_indices if i > 0]
sensitive = sorted(set(attn_indices + pre_attn_indices))
print(f'\nAttention layers (*):       {attn_indices}')
print(f'Pre-attention Mamba layers: {pre_attn_indices}')
print(f'All sensitive layers (12):  {sensitive}')

# Scan all modules, keep Linear ones in sensitive layers
all_mods = dict(model.named_modules())
target_modules = []
skipped = {}

for name, mod in all_mods.items():
    if 'lora' in name.lower():
        continue
    # Must be in a backbone layer
    m = _re.search(r'backbone\.layers\.(\d+)', name)
    if not m:
        continue
    layer_idx = int(m.group(1))
    if layer_idx not in sensitive:
        continue
    # Must be Linear (Conv1d has groups=6144, incompatible with LoRA)
    if not isinstance(mod, torch.nn.Linear):
        typ = type(mod).__name__
        skipped[typ] = skipped.get(typ, 0) + 1
        continue
    # Skip router/gate weights (sparse routing)
    if 'router' in name.lower() or 'gate' in name.lower():
        continue
    # Skip routable experts (index >= 2, sparse traffic)
    expert_m = _re.search(r'experts\.(\d+)', name)
    if expert_m and int(expert_m.group(1)) >= 2:
        continue
    target_modules.append(name)

print(f'\nLoRA target modules: {len(target_modules)}')
print(f'Skipped non-Linear: {skipped}')

# Show breakdown
mixer_count = sum(1 for n in target_modules if '.mixer.' in n and 'expert' not in n)
expert_count = sum(1 for n in target_modules if 'expert' in n)
other_count = len(target_modules) - mixer_count - expert_count
print(f'Breakdown: mixer={mixer_count}, shared_expert={expert_count}, other={other_count}')

assert len(target_modules) >= 20, f'Only {len(target_modules)} targets found — check layer discovery!'


Architecture pattern: MEMEM*EMEMEM*EMEMEM*EMEMEM*EMEMEM*EMEMEMEM*EMEMEMEME
Pattern length: 52 layers

Attention layers (*):       [5, 12, 19, 26, 33, 42]
Pre-attention Mamba layers: [4, 11, 18, 25, 32, 41]
All sensitive layers (12):  [4, 5, 11, 12, 18, 19, 25, 26, 32, 33, 41, 42]

LoRA target modules: 36
Skipped non-Linear: {'NemotronHBlock': 12, 'NemotronHRMSNorm': 12, 'NemotronHMamba2Mixer': 6, 'SiLUActivation': 6, 'Conv1d': 6, 'MambaRMSNormGated': 6, 'NemotronHAttention': 6}
Breakdown: mixer=36, shared_expert=0, other=0


## Step 6: Apply LoRA

In [6]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

print(f'Applying LoRA (rank={LORA_RANK}) to {len(target_modules)} modules...')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You should see 50-150 target modules and ~20-50M trainable params
# If you see 880M params or 3+ GB adapter, too many layers are targeted


Applying LoRA (rank=32) to 36 modules...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 7,532,544 || all params: 31,585,469,888 || trainable%: 0.0238


## Step 7: Prepare Training Data

Each training example is formatted as a chat conversation:
- **User**: the puzzle prompt + instruction to use `\boxed{}`
- **Assistant**: `\boxed{answer}`

At inference time, the model has 7,680 tokens to generate. Its native `<think>` reasoning will fill in the gap — we don't need to provide reasoning traces.

In [7]:
train_df = pd.read_csv(DATA_PATH)
print(f'Training examples: {len(train_df)}')
print(f'\nColumns: {list(train_df.columns)}')
print(f'\nSample:')
print(f'  Prompt: {train_df["prompt"].iloc[0][:150]}...')
print(f'  Answer: {train_df["answer"].iloc[0]}')

# Format each example as a chat conversation
def format_example(row):
    user_msg = row['prompt'] + '\nPut your final answer inside \\boxed{}.'
    assistant_msg = f'\\boxed{{{row["answer"]}}}'
    try:
        return tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_msg},
             {'role': 'assistant', 'content': assistant_msg}],
            tokenize=False, add_generation_prompt=False
        )
    except:
        # Fallback manual template
        return (
            f'<|im_start|>system\n<|im_end|>\n'
            f'<|im_start|>user\n{user_msg}<|im_end|>\n'
            f'<|im_start|>assistant\n<think></think>{assistant_msg}<|im_end|>'
        )

texts = [format_example(row) for _, row in train_df.iterrows()]
random.shuffle(texts)
dataset = Dataset.from_dict({'text': texts})

print(f'\nDataset ready: {len(dataset)} examples')
print(f'\nFormatted sample (first 400 chars):')
print(texts[0][:400])


Training examples: 9500

Columns: ['id', 'prompt', 'answer']

Sample:
  Prompt: In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotatio...
  Answer: 10010111

Dataset ready: 9500 examples

Formatted sample (first 400 chars):
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01111100 -> 00000011
00010100 -> 00000000
10101101 -> 00000101
11110000 -> 00000111
01101111 


## Step 8: Train (your code here)

The setup is complete. Add your training logic below. The example uses `SFTTrainer` from TRL — uncomment it to run.

In [8]:
# ════════════════════════════════════════════════════════
# YOUR TRAINING CODE HERE
# ════════════════════════════════════════════════════════
# Everything above is ready. The model has LoRA applied,
# the dataset is formatted, the tokenizer is set.
#
# Example using SFTTrainer:
#
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=5e-5,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    warmup_steps=10,
    save_strategy='no',
    report_to='none',
    dataset_text_field='text',
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)
trainer.train()

print('Training code goes here. Uncomment the block above to train.')
print('At ~13 sec/step with grad_accum=8, 1 epoch on 9500 examples = ~4.3 hours.')


Adding EOS to train dataset:   0%|          | 0/9500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Step,Training Loss
5,42.672458
10,43.553384
15,39.921219
20,37.490823
25,38.450616
30,32.218845
35,31.025781
40,29.492609
45,25.278462
50,22.425421


Training code goes here. Uncomment the block above to train.
At ~13 sec/step with grad_accum=8, 1 epoch on 9500 examples = ~4.3 hours.


## Step 9: Quick Validation

In [9]:
model.eval()
val_df = pd.read_csv(DATA_PATH).head(5)

print('=== Validation (5 examples) ===')
correct = 0
for _, row in val_df.iterrows():
    prompt = row['prompt'] + '\nPut your final answer inside \\boxed{}.'
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=1500).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    match = re.search(r'\\boxed\{([^}]+)\}', response)
    predicted = match.group(1).strip() if match else response.strip()[:50]
    expected = str(row['answer']).strip()

    try:
        hit = abs(float(predicted) - float(expected)) / max(abs(float(expected)), 1e-9) < 1e-3
    except:
        hit = predicted == expected

    if hit:
        correct += 1
    print(f'{"OK" if hit else "MISS"}  expected={expected}  got={predicted}')

print(f'\nAccuracy: {correct}/5')


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Validation (5 examples) ===
MISS  expected=10010111  got=We need to determine the output for the 8-bit bina
MISS  expected=01000011  got=We need to determine the output for the 8-bit bina
MISS  expected=cat imagines book  got=The student writes the following text: wizard draw
MISS  expected=XXXVIII  got=The assistant has converted the number 38 into the
MISS  expected=wizard creates secret  got=the colorful rabbit reads

Accuracy: 0/5


## Step 10: Save and Package Submission

In [10]:
# Save the LoRA adapter
# After training, save with: trainer.model.save_pretrained(OUTPUT_DIR)
# For this demo (no training), save the base LoRA adapter:
model.save_pretrained(OUTPUT_DIR)

print('Adapter files:')
for f in os.listdir(OUTPUT_DIR):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        size = os.path.getsize(fp)
        print(f'  {f} ({size/1024:.1f} KB)')

# Package into submission.zip (required by competition)
zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, fname)  # files at zip root

# Verify
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f'\nsubmission.zip contents: {names}')
    assert 'adapter_config.json' in names, 'Missing adapter_config.json!'

print(f'submission.zip size: {os.path.getsize(zip_path)/1024/1024:.1f} MB')
print('\nReady to submit!')


Adapter files:
  adapter_config.json (1.3 KB)
  README.md (5.2 KB)
  adapter_model.safetensors (29433.4 KB)

submission.zip contents: ['adapter_config.json', 'README.md', 'adapter_model.safetensors']
submission.zip size: 26.6 MB

Ready to submit!


## Approximate Timing (GPU RTX Pro 6000)

| Phase | Time |
|---|---|
| Environment setup + pip install | ~1 min |
| Model loading | ~8 min |
| Layer discovery + LoRA setup | ~1 min |
| Data formatting (9,500 examples) | ~1 min |
| **Training (per step, grad_accum=8)** | **~13 seconds** |
| Validation (5 examples) | ~3 min |
| Save adapter + zip | ~1 min |

### Training time estimates

| Epochs | Steps | Training Time | Total Runtime |
|---|---|---|---|
| 1 | ~1,188 | ~4.3 hours | ~4.5 hours |
| 2 | ~2,375 | ~8.6 hours | ~8.8 hours |
| 3 | ~3,564 | ~12.9 hours | ~13 hours |

With the 9-hour Kaggle GPU budget, **1 epoch fits comfortably**. 2 epochs is tight. Use the `SmartStopCallback` from the competition notebook to auto-stop if loss plateaus or time runs out.

> **Adapter size:** With sensitive-layer targeting, expect **~26 MB**. If you see 3+ GB, you're targeting too many layers.

## Next Steps

This starter gets you a valid submission. To improve your score:

1. **Add deterministic solvers** — Roman numerals, gravity (d=½gt²), and unit conversion can be solved with Python at 99%+ accuracy. Use solver outputs as training traces inside `<think>` tags.

2. **Target hard puzzle types** — Bit manipulation and symbol transformation are where the competition is won. The model already handles easy types well.

3. **Increase epochs** — With the 9-hour time budget and ~26 MB adapter, you can train for 3-5 epochs on all 9,500 examples.

4. **Try GRPO** — `trl` includes `GRPOTrainer` for reinforcement learning with reward functions. Reward correct `\boxed{}` answers to refine reasoning beyond SFT.

5. **Response-only loss** — Use `DataCollatorForCompletionOnlyLM` from trl to compute loss only on the answer, not the prompt.

**Key constraints:**
- Max LoRA rank: 32
- Inference uses vLLM with temperature=0.0, max_tokens=7680
- Only `adapter_config.json` + `adapter_model.safetensors` in submission.zip